# OMR demo inference server (Colab)

Loads fine-tuned LoRA adapter `NathanZ721/omr-monophonic` on top of `Qwen/Qwen2-VL-2B-Instruct` and exposes a public Flask `/infer` endpoint via ngrok for the local demo app.

**Steps**
1. Select a GPU runtime (T4 is enough).
2. Run cell 1 once — it repairs Pillow/torchao and **restarts Colab automatically**. After reconnect, run cell 1 again (it should finish without restarting), then continue.
3. Paste your ngrok authtoken in cell 4 (https://dashboard.ngrok.com/get-started/your-authtoken).
4. Run the remaining cells.
5. Copy the printed public URL into the local demo UI / `COLAB_INFER_URL`.

If you already hit a `PIL._typing._Ink` ImportError in this runtime, use **Runtime → Restart session** (or Disconnect and delete runtime), then start from cell 1 with the updated notebook.

In [ ]:
# @title 1) Install dependencies (Pillow repair + one guarded restart)
# Do not import PIL, transformers, or peft before this cell finishes and Colab restarts.
import importlib.machinery
import importlib.metadata
import os
import signal
import subprocess
import sys
from pathlib import Path

PILLOW_VERSION = "12.3.0"
_SETUP_REVISION = 1
_SETUP_MARKER = Path(
    f"/content/.omr_demo_server_setup_v{_SETUP_REVISION}_pillow_"
    f"{PILLOW_VERSION.replace('.', '_')}_no_torchao_ready"
)
_RESTART_GUARD = Path(f"/content/.omr_demo_server_setup_v{_SETUP_REVISION}_restart_requested")


def _run_pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", *args])


def _installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def _torchao_files_importable() -> bool:
    return importlib.machinery.PathFinder.find_spec("torchao") is not None


def _setup_is_valid() -> bool:
    return (
        _installed_version("pillow") == PILLOW_VERSION
        and _installed_version("torchao") is None
        and not _torchao_files_importable()
        and _installed_version("transformers") is not None
        and _installed_version("accelerate") is not None
        and _installed_version("peft") is not None
        and _installed_version("qwen-vl-utils") is not None
        and _installed_version("safetensors") is not None
        and _installed_version("flask") is not None
        and _installed_version("pyngrok") is not None
    )


if not (_SETUP_MARKER.exists() and _setup_is_valid()):
    if _RESTART_GUARD.exists():
        raise RuntimeError(
            "Dependency setup is still inconsistent after the one guarded restart. "
            "Use Runtime -> Disconnect and delete runtime, then run again."
        )

    print(f"Installing dependencies and a clean Pillow {PILLOW_VERSION} ...", flush=True)
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "-U",
        "transformers>=4.49.0,<5.0.0",
        "accelerate",
        "peft",
        "qwen-vl-utils",
        "safetensors",
        "flask",
        "pyngrok",
    )
    # Force a clean Pillow last so torchvision/transformers do not leave mixed PIL modules.
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        f"pillow=={PILLOW_VERSION}",
    )
    _run_pip("uninstall", "-q", "-y", "torchao")

    if not _setup_is_valid():
        raise RuntimeError(
            "Dependency repair failed before restart: "
            f"pillow={_installed_version('pillow')}, "
            f"torchao={_installed_version('torchao')}, "
            f"torchao_importable={_torchao_files_importable()}"
        )

    _SETUP_MARKER.write_text(
        f"revision={_SETUP_REVISION}\npillow={PILLOW_VERSION}\ntorchao=absent\n",
        encoding="utf-8",
    )
    _RESTART_GUARD.write_text("automatic restart requested once\n", encoding="utf-8")
    print(
        "Setup complete. Colab will restart once. After reconnect, rerun from this cell "
        "(cell 1), then continue. The guard prevents a restart loop.",
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Automatic restart failed. Restart the session manually, then rerun from cell 1.")

# Import checks only after the clean restarted process.
import PIL
from PIL import Image, ImageDraw, ImageText
from PIL._typing import _Ink

import peft
import torch
import transformers

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"peft={peft.__version__}")
print(f"pillow={PIL.__version__}")
print("torchao=absent; Pillow ImageDraw/ImageText imports: OK")

In [ ]:
# @title 2) Load Qwen2-VL-2B + OMR LoRA adapter
import re

import torch
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
ADAPTER_ID = "NathanZ721/omr-monophonic"
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 768 * 768

SYSTEM_PROMPT = "You are an optical music recognition system. Return only PrIMuS semantic tokens."
USER_PROMPT = (
    "Transcribe this monophonic sheet-music image into PrIMuS semantic format. "
    "Return only one space-separated token string with no explanation or markdown."
)

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_ID, is_trainable=False)
model.to(device).eval()

print(f"Loaded {BASE_MODEL_ID} + {ADAPTER_ID} on {device} ({dtype})")

In [ ]:
# @title 3) Inference helpers
from io import BytesIO


def make_messages():
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]


def clean_prediction(text: str) -> str:
    text = re.sub(r"```(?:\w+)?", "", text or "").replace("```", "")
    return " ".join(text.split())


def predict_pil(image: Image.Image) -> str:
    image = image.convert("RGB")
    image.load()
    prompt_text = processor.apply_chat_template(
        make_messages(),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[prompt_text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    generated = output_ids[:, prompt_length:]
    return clean_prediction(
        processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    )


def predict_bytes(image_bytes: bytes) -> str:
    with Image.open(BytesIO(image_bytes)) as source:
        return predict_pil(source)


print("Inference helpers ready.")

In [ ]:
# @title 4) Configure ngrok + start Flask /infer server
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
SERVER_PORT = 8000  # @param {type:"integer"}

import threading

from flask import Flask, jsonify, request
from pyngrok import ngrok

if not NGROK_AUTHTOKEN.strip():
    raise ValueError(
        "Set NGROK_AUTHTOKEN from https://dashboard.ngrok.com/get-started/your-authtoken"
    )

ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

app = Flask(__name__)


@app.after_request
def add_cors_headers(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization, ngrok-skip-browser-warning"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    return response


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"ok": True, "model": BASE_MODEL_ID, "adapter": ADAPTER_ID})


@app.route("/infer", methods=["POST", "OPTIONS"])
def infer():
    if request.method == "OPTIONS":
        return ("", 204)
    if "image" not in request.files:
        return jsonify({"error": "Expected multipart field 'image'"}), 400
    upload = request.files["image"]
    image_bytes = upload.read()
    if not image_bytes:
        return jsonify({"error": "Empty image upload"}), 400
    try:
        semantic = predict_bytes(image_bytes)
    except Exception as exc:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return jsonify({"error": type(exc).__name__, "detail": str(exc)}), 500
    return jsonify({"semantic": semantic})


def _run_flask():
    app.run(host="0.0.0.0", port=SERVER_PORT, threaded=True, use_reloader=False)


thread = threading.Thread(target=_run_flask, daemon=True)
thread.start()

tunnel = ngrok.connect(SERVER_PORT, "http")
public_url = tunnel.public_url.rstrip("/")
print("=" * 72)
print("Colab OMR server is live")
print(f"Public URL:  {public_url}")
print(f"Health:      {public_url}/health")
print(f"Infer:       POST {public_url}/infer  (multipart field: image)")
print("Paste the Public URL into the local demo app as COLAB_INFER_URL.")
print("=" * 72)